# Testing pipeline

In [13]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score

from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from prophet import Prophet

### Dataset loading

In [14]:
df = pd.read_csv("datasets/dataset1_forecastets.csv")

In [15]:
df.head()

,transaction_id,transaction_date,currency,description,vendor_name,gst_applicable,gst_slab,itc_eligible,category_label,is_anomaly,amount,dow
0,TXN0000001,01-04-2024,INR,service fees,IKEA,True,exempt,FALSE,Office Supplies,0,5567.844445,0
1,TXN0000002,01-04-2024,INR,upi awfis rcpt/2404/002 fy24 - ent ex# ref 6160,Awfis,True,12%,unknown,Rent,0,73161.312250,0
2,TXN0000003,01-04-2024,INR,bill paid,ZOMATO,True,5%,FALSE,Office Supplies,0,2339.523467,0
3,TXN0000004,01-04-2024,INR,!UPI UBER INDIA PVT LTD TAXINV/2501/924 2 - ex...,OFFICE DEPOT Pvt Ltd,True,exempt,unknown,Office Supplies,0,3351.408985,0
4,TXN0000005,01-04-2024,INR,auto debit,AWFIS INDIA PVT LTD,True,18%,TRUE,Rent,0,58873.821700,0


In [16]:


# Parse dates (dd-mm-YYYY)
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")

# Drop rows with critical missing values
df = df.dropna(subset=["transaction_date","amount"])
df = df.dropna(subset=["description", "category_label"])

# Vendor name: fill missing instead of dropping whole rows
df["vendor_name"] = df["vendor_name"].fillna("UNKNOWN_VENDOR")

# Keep only INR if needed
df = df[df["currency"].astype(str).str.upper() == "INR"].copy()


In [17]:
df.head()

,transaction_id,transaction_date,currency,description,vendor_name,gst_applicable,gst_slab,itc_eligible,category_label,is_anomaly,amount,dow
0,TXN0000001,2024-01-04,INR,service fees,IKEA,True,exempt,FALSE,Office Supplies,0,5567.844445,0
1,TXN0000002,2024-01-04,INR,upi awfis rcpt/2404/002 fy24 - ent ex# ref 6160,Awfis,True,12%,unknown,Rent,0,73161.312250,0
2,TXN0000003,2024-01-04,INR,bill paid,ZOMATO,True,5%,FALSE,Office Supplies,0,2339.523467,0
3,TXN0000004,2024-01-04,INR,!UPI UBER INDIA PVT LTD TAXINV/2501/924 2 - ex...,OFFICE DEPOT Pvt Ltd,True,exempt,unknown,Office Supplies,0,3351.408985,0
4,TXN0000005,2024-01-04,INR,auto debit,AWFIS INDIA PVT LTD,True,18%,TRUE,Rent,0,58873.821700,0


### Preprocessing and cleaning

In [18]:
def clean_text(s: str) -> str:
    """Lowercase, remove invoice IDs etc. and non-alphanumeric noise."""
    if not isinstance(s, str):
        return ""
    s = s.lower()
    noise_tokens = [
        r"\bfy\d{2}\b",          # fy24, fy25
        r"\bq[1-4]\b",           # q1, q2
        r"\binv/?\d*\b",         # inv/2404/001
        r"\bbill/?\d*\b",        # bill/...
        r"\brcpt/?\d*\b",        # rcpt/...
        r"\btaxinv/?\d*\b",      # taxinv/...
        r"\bref\b\s*\d+",        # ref 1234
        r"\badv\b",
        r"\bsubs\b",
        r"\bgst\b",
    ]
    for pat in noise_tokens:
        s = re.sub(pat, " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def clean_vendor(v: str) -> str:
    """Normalize vendor name to reduce variation (PVT, LTD, INDIA, etc.)."""
    if not isinstance(v, str):
        return ""
    v = v.upper()
    v = re.sub(r"\bPVT\.?\b|\bLTD\.?\b|\bLIMITED\b|\bINDIA\b", " ", v)
    v = re.sub(r"[^A-Z0-9\s]", " ", v)
    v = re.sub(r"\s+", " ", v).strip()
    return v

df["description_clean"] = df["description"].apply(clean_text)
df["vendor_clean"] = df["vendor_name"].apply(clean_vendor)

# Time features
df["month"] = df["transaction_date"].dt.month
df["dow"] = df["transaction_date"].dt.dayofweek  # 0=Monday

# For modeling: combine text & vendor text to a single text field
df["text_combo"] = (
    df["description_clean"].fillna("") + " " + df["vendor_clean"].fillna("")
)

### Modeling (TFIDF + XGBoost)

In [19]:
TEXT_COL = "text_combo"
NUM_COLS = ["amount", "month", "dow"]
TARGET_COL = "category_label"

X = df[[TEXT_COL] + NUM_COLS]
y = df[TARGET_COL]

# Encode labels to integers for XGBoost
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

# ColumnTransformer: TF-IDF on text, passthrough numeric
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
)

preprocess = ColumnTransformer(
    transformers=[
        ("text", tfidf, TEXT_COL),
        ("num", "passthrough", NUM_COLS),
    ],
    remainder="drop",
)

xgb_clf = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    max_depth=8,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
)

pipe_xgb = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("clf", xgb_clf),
    ]
)

# Train
pipe_xgb.fit(X_train, y_train)

# Predict once
y_pred_enc = pipe_xgb.predict(X_test)
y_test_str = le.inverse_transform(y_test)
y_pred_str = le.inverse_transform(y_pred_enc)

print("=== TF-IDF (1–2 grams) + XGBoost (multi-class) ===")
print(classification_report(y_test_str, y_pred_str))
print("Macro F1:",
      f1_score(y_test_str, y_pred_str, average="macro"))

# Attach predictions to the corresponding test rows
df_test = df.loc[X_test.index].copy()
df_test["pred_category"] = y_pred_str


=== TF-IDF (1–2 grams) + XGBoost (multi-class) ===
                 precision    recall  f1-score   support

Exempt Services       0.91      1.00      0.95        21
    IT Services       0.90      0.88      0.89       173
          Meals       0.89      0.89      0.89        45
Office Supplies       0.95      0.94      0.95       102
           Rent       1.00      0.97      0.99       119
       Software       0.98      1.00      0.99        88
       Training       0.76      0.79      0.77        67
         Travel       0.97      0.96      0.97        79
      Utilities       0.90      0.92      0.91       102

       accuracy                           0.92       796
      macro avg       0.92      0.93      0.92       796
   weighted avg       0.93      0.92      0.92       796

Macro F1: 0.9229809882613207


### Tax Engine

In [20]:
# Simple rule table 
gst_rules = {
    "Meals":           {"gst_rate": 5,  "itc_eligible": "No"},
    "Travel":          {"gst_rate": 5,  "itc_eligible": "No"},
    "Rent":            {"gst_rate": 18, "itc_eligible": "Yes"},
    "IT Services":     {"gst_rate": 18, "itc_eligible": "Yes"},
    "Software":        {"gst_rate": 18, "itc_eligible": "Yes"},
    "Office Supplies": {"gst_rate": 12, "itc_eligible": "Yes"},
    "Utilities":       {"gst_rate": 18, "itc_eligible": "Yes"},
    "Training":        {"gst_rate": 18, "itc_eligible": "Unknown"},
    "Exempt Services": {"gst_rate": 0,  "itc_eligible": "No"},
}

def attach_tax_labels(df_in, category_col: str = "pred_category"):
    """Attach gst_rate and itc_eligible columns based on predicted category."""
    df_out = df_in.copy()
    df_out["gst_rate_pred"] = df_out[category_col].map(
        lambda c: gst_rules.get(c, {}).get("gst_rate", 0)
    )
    df_out["itc_eligible_pred"] = df_out[category_col].map(
        lambda c: gst_rules.get(c, {}).get("itc_eligible", "Unknown")
    )
    return df_out

df_test_tax = attach_tax_labels(df_test, category_col="pred_category")

print("\n=== Sample tax-engine output on test set ===")
print(
    df_test_tax[
        ["transaction_id", "transaction_date", "amount",
         "pred_category", "gst_rate_pred", "itc_eligible_pred"]
    ].head()
)

# Optional: compare with original gst_slab / itc_eligible to see mismatches

def parse_gst_slab(s: str) -> float:
    """
    Convert gst_slab like '18%' or '5%' to float 18.0 / 5.0.
    Map 'exempt' or anything non-numeric to 0.0 for comparison.
    """
    if not isinstance(s, str):
        return 0.0
    s = s.strip().lower()
    if s == "exempt":
        return 0.0
    s = s.replace("%", "")
    try:
        return float(s)
    except ValueError:
        return 0.0

df_test_tax["gst_slab_numeric"] = df_test_tax["gst_slab"].apply(parse_gst_slab)

df_test_tax["gst_slab_mismatch"] = (
    df_test_tax["gst_slab_numeric"].fillna(0.0)
    != df_test_tax["gst_rate_pred"].fillna(0.0)
)

print(
    "\nGST slab mismatch rate (test set):",
    df_test_tax["gst_slab_mismatch"].mean()
)


=== Sample tax-engine output on test set ===
     transaction_id transaction_date        amount    pred_category  \
2        TXN0000003       2024-01-04   2339.523467            Meals   
6813     TXN0001814       2025-08-08   8644.710225  Office Supplies   
5569     TXN0000570       2025-07-05   2711.229964           Travel   
2534     TXN0002535       2024-04-10  17351.832540        Utilities   
870      TXN0000871       2024-03-06   6490.995696           Travel   

      gst_rate_pred itc_eligible_pred  
2                 5                No  
6813             12               Yes  
5569              5                No  
2534             18               Yes  
870               5                No  

GST slab mismatch rate (test set): 0.46984924623115576


### Anomaly detection (IsoForest + LOF)

In [21]:


# Build numeric + time + vendor-context features
df["day_of_week"] = df["transaction_date"].dt.dayofweek
df["day_of_month"] = df["transaction_date"].dt.day
df["month"] = df["transaction_date"].dt.month

# Use noisy amount if present, otherwise amount
amount_col = "amount_noisy" if "amount_noisy" in df.columns else "amount"
df["log_amount"] = np.log1p(df[amount_col])

vendor_stats = df.groupby("vendor_name")[amount_col].agg(
    vendor_mean="mean",
    vendor_std="std",
    vendor_count="count"
).reset_index()

df = df.merge(vendor_stats, on="vendor_name", how="left")

features_anom = [
    "log_amount",
    "day_of_week",
    "day_of_month",
    "month",
    "vendor_mean",
    "vendor_std",
    "vendor_count",
]

X_anom = df[features_anom].fillna(0.0).values

# --- Isolation Forest (global anomalies) ---
iso = IsolationForest(
    n_estimators=300,
    contamination=0.01,  # ~1% anomalies
    random_state=42,
    n_jobs=-1,
)

iso.fit(X_anom)

df["iso_score"] = iso.decision_function(X_anom)   # higher = more normal
df["iso_flag"] = iso.predict(X_anom)              # -1 = anomaly, 1 = normal

# --- Local Outlier Factor (local anomalies) ---
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.01,
    novelty=False
)

lof_labels = lof.fit_predict(X_anom)   # -1 = outlier
lof_scores = lof.negative_outlier_factor_

df["lof_flag"] = lof_labels
df["lof_score"] = lof_scores           # more negative = more anomalous

# Inspect top anomalies
print("\n=== Top 10 Isolation Forest anomalies ===")
print(
    df[df["iso_flag"] == -1]
    .sort_values("iso_score")
    [["transaction_id", "transaction_date", "vendor_name",
      amount_col, "category_label", "iso_score"]]
    .head(10)
)

print("\n=== Top 10 LOF anomalies ===")
print(
    df[df["lof_flag"] == -1]
    .sort_values("lof_score")
    [["transaction_id", "transaction_date", "vendor_name",
      amount_col, "category_label", "lof_score"]]
    .head(10)
)



=== Top 10 Isolation Forest anomalies ===
     transaction_id transaction_date          vendor_name        amount  \
2366     TXN0000933       2025-05-06                  HCL  3.056979e+06   
2908     TXN0002261       2025-11-09             INDIQUBE  9.616710e+05   
2911     TXN0002265       2025-11-09             INDIQUBE  8.188702e+05   
3119     TXN0002992       2025-01-11               WEWORK  4.652488e+05   
3124     TXN0002997       2025-01-11                  HCL  9.006138e+04   
3114     TXN0002719       2025-12-10                  HCL  2.627659e+05   
3483     TXN0003888       2026-03-01                  HCL  2.131294e+04   
489      TXN0001245       2024-01-07  AWFIS INDIA PVT LTD  5.375571e+05   
1383     TXN0003444       2024-08-12                  HCL  2.000396e+04   
3528     TXN0003933       2026-06-01                  HCL  2.412811e+04   

     category_label  iso_score  
2366    IT Services  -0.032142  
2908           Rent  -0.023862  
2911           Rent  -0.022085  

### Forecasting

In [22]:
# Aggregate to daily total amount (full dataset, not only test)
daily = (
    df.groupby("transaction_date", as_index=False)["amount"]
      .sum()
      .rename(columns={"transaction_date": "ds", "amount": "y"})
)


# Time series aggregation & forecast


from statsmodels.tsa.arima.model import ARIMA

# Aggregate to daily total amount (full dataset, not only test)
daily = (
    df.groupby("transaction_date", as_index=False)["amount"]
      .sum()
      .rename(columns={"transaction_date": "ds", "amount": "y"})
)

daily = daily.sort_values("ds").dropna(subset=["ds", "y"]).reset_index(drop=True)

if len(daily) < 5:
    print("Not enough points for forecasting, skipping time-series modelling.")
else:
    # Try Prophet first, but do NOT let it crash the notebook
    used_method = None
    forecast_df = None

    try:
        from prophet import Prophet

        m = Prophet(
            yearly_seasonality=False,
            weekly_seasonality=True,
            daily_seasonality=False,
            changepoint_prior_scale=0.05,
        )

        m.fit(daily)

        future = m.make_future_dataframe(periods=30)
        forecast = m.predict(future)

        forecast_df = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]
        used_method = "Prophet"

    except RuntimeError as e:
        print("Prophet failed on this series, falling back to ARIMA.")
        # ARIMA on the same y series
        y_series = daily.set_index("ds")["y"]
        # Simple ARIMA(1,1,1) as a baseline; you can tune if needed
        arima_model = ARIMA(y_series, order=(1, 1, 1)).fit()

        # Build future index for 30 days
        last_date = daily["ds"].max()
        future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                                     periods=30, freq="D")
        arima_forecast = arima_model.forecast(steps=30)

        forecast_df = pd.DataFrame({
            "ds": future_dates,
            "yhat": arima_forecast,
        })
        used_method = "ARIMA(1,1,1)"

    if forecast_df is not None:
        print(f"\n=== {used_method} daily amount forecast (last few rows) ===")
        print(forecast_df.tail())





17:54:32 - cmdstanpy - INFO - Chain [1] start processing
17:54:32 - cmdstanpy - INFO - Chain [1] done processing
17:54:32 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
Optimization terminated abnormally. Falling back to Newton.
17:54:32 - cmdstanpy - INFO - Chain [1] start processing
17:54:32 - cmdstanpy - INFO - Chain [1] done processing
17:54:32 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


Prophet failed on this series, falling back to ARIMA.

=== ARIMA(1,1,1) daily amount forecast (last few rows) ===
            ds           yhat
313 2026-12-29  605353.129839
314 2026-12-30  605353.129839
315 2026-12-31  605353.129839
316 2027-01-01  605353.129839
317 2027-01-02  605353.129839


C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\USER\AppData\Roaming\Python\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an

### Insights

In [23]:
y_all_pred_enc = pipe_xgb.predict(X[[TEXT_COL] + NUM_COLS])
y_all_pred_str = le.inverse_transform(y_all_pred_enc)
df_all = df.copy()
df_all["pred_category"] = y_all_pred_str

cat_summary = (
    df_all.groupby("pred_category")["amount"]
          .agg(["count", "sum", "mean"])
          .sort_values("sum", ascending=False)
)

print("\n=== Spend by predicted category (full data) ===")
print(cat_summary)

print("\nKey insights:")
print("- Top categories by spend highlight where GST outflow and ITC claims will concentrate.")
print("- Mismatch rate between predicted GST rate and original gst_slab shows potential mis-classifications or rule differences.")
print("- Daily forecast gives an indication of expected near-term cash outflows for planning.")


=== Spend by predicted category (full data) ===
                 count           sum           mean
pred_category                                      
Rent               590  7.850509e+07  133059.467187
IT Services        858  4.657644e+07   54284.890403
Training           336  1.322386e+07   39356.730561
Utilities          515  9.807544e+06   19043.773918
Software           441  5.221721e+06   11840.636201
Office Supplies    507  4.396049e+06    8670.708296
Travel             396  3.096532e+06    7819.524874
Meals              224  9.535496e+05    4256.918059
Exempt Services    110  7.960682e+05    7236.983560

Key insights:
- Top categories by spend highlight where GST outflow and ITC claims will concentrate.
- Mismatch rate between predicted GST rate and original gst_slab shows potential mis-classifications or rule differences.
- Daily forecast gives an indication of expected near-term cash outflows for planning.
